<a href="https://colab.research.google.com/github/charansai12322/weatherapp/blob/main/weatherapp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install groq

In [2]:
from google.colab import userdata
from groq import Groq

client = Groq(
  api_key=userdata.get('groq')
)

response = client.chat.completions.create(
  messages=[ {
    "role": "user",
    "content": "What is the current weather in hyderabad",
  }], model="openai/gpt-oss-120b",
)
print(response.choices[0].message.content)

I’m not able to retrieve live weather data. For the most up‑to‑date conditions in Hyderabad, you can check a reliable weather service or app such as:

- **Weather.com** – search for “Hyderabad, India”
- **AccuWeather** – same search
- **Google** – type “Hyderabad weather” and the current temperature, humidity, and forecast will appear at the top of the results
- **Your phone’s weather app** – usually pulls the latest data automatically

These sources will give you real‑time temperature, chance of rain, wind speed, air quality, and any short‑term forecasts. If you need help interpreting the forecast or planning activities around the weather, feel free to let me know!


In [3]:
import json
import requests
from google.colab import userdata

def get_weather(location):
  api_key = userdata.get('WEATHER_API_KEY')
  url = f"http://api.openweathermap.org/data/2.5/weather?q={location}&units=metric&appid={api_key}"

  response = requests.get(url)
  data = response.json()
  if data.get("cod") == 200:
    return json.dumps({
      "location": location,
      "temperature": data["main"]["temp"],
      "description": data["weather"][0]["description"]
    })
  else:
    return json.dumps({"error": "Oops! Something went wrong."})

In [4]:
tools = [
  {
    "type": "function",
    "function": {
      "name": "get_weather",
      "description": "Get current weather for a city",
      "parameters": {
        "type": "object",
        "properties": {
          "location": {
            "type": "string",
            "description": "City name like Mumbai, London"
          }},
        "required": ["location"]
      }}}
]

In [5]:
llm_messages = [
  {
    "role": "system",
    "content": "You are a weather assistant. Use get_weather function when asked about weather."
  },
  {
    "role": "user",
    "content": "What's the weather in Mumbai?"
  }
]

response = client.chat.completions.create(
  model="openai/gpt-oss-120b",
  messages=llm_messages,
  tools=tools,
  tool_choice="auto"
)

response_message = response.choices[0].message

if response_message.tool_calls:
  tool_call = response_message.tool_calls[0]
  arguments = json.loads(tool_call.function.arguments)
  location = arguments['location']
  weather_data = get_weather(location)

  llm_messages.append(response_message)

  llm_messages.append({
      "role": "tool",
      "tool_call_id": tool_call.id,
      "content": weather_data
  })


In [6]:

  final_response = client.chat.completions.create(
      messages = llm_messages,
      model = "openai/gpt-oss-120b",
      tools = tools,
      tool_choice = "auto"
  )

  print(final_response.choices[0].message.content)

The current weather in Mumbai is **31.99 °C** with a **clear sky**.


In [7]:
print("=== ACTUAL WEATHER ===")
print(weather_data)

print("\n=== MODEL RESPONSE ===")
print(final_response.choices[0].message.content)

=== ACTUAL WEATHER ===
{"location": "Mumbai", "temperature": 31.99, "description": "clear sky"}

=== MODEL RESPONSE ===
The current weather in Mumbai is **31.99 °C** with a **clear sky**.
